# 9.9 最大 softmax 機率很高就可靠嗎？


模型在三個候選中給第一項99.99%的概率，看起來非常肯定。但如果第一項原本就是錯的，分數變尖並沒有讓知識變多。像投票時把每個人的票複製十次，勝者不變，票數差卻變大。本節分清「候選之間的相對分數」與「回答是否正確」。

前置是[1.7從分數到機率](https://birdhackor.github.io/tiny-perceptron-vlm/1.7.html)。回顧：logits是還未變成機率的分數，softmax把它們轉換為總和1的比例，最大比例常被拿來當信心。它比較的只是在場候選，不會自動查閱外部標準答案，也不保證最高項代表事實。模型說「我很有信心」又是一段文字，與這項數值也不是同一件事。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

scores = torch.tensor([[2.0, 1.0, 0.0]])
correct_index = 1
for scale in [1.0, 10.0]:
    probability = (scores * scale).softmax(-1)
    confidence, prediction = probability.max(-1)
    print(
        "倍率",
        scale,
        "信心",
        round(confidence.item(), 4),
        "選擇",
        prediction.item(),
        "正確",
        prediction.item() == correct_index,
    )

輸入三項分數2、1、0，另用`correct_index=1`設第二項是標準答案。索引從0起，因此最高分的第一項是錯誤候選。程式只改變整體倍率，`softmax(-1)`在最後的候選軸做轉換，`max(-1)`同時取最大比例與其索引。`round`讓小數便於比較。

倍率1時輸出信心約0.6652、選擇0、正確False；倍率10時輸出信心約1.0、選擇仍0、正確仍False。由於四舍五入，看起來恰好1.0，內部其實仍略小於1。兩次沒有訓練、沒有加入任何事實，變化只來自分數尺度。這正是高信心錯誤的可追蹤例子。

有些系統用temperature溫度調整分數：先除以一個正數再做softmax，溫度更大通常使分布更平。它可以幫助統計上的信心與正確率更接近，叫校準，但不能把錯誤的最高候選變成已學會的新知識。溫度與抽樣時控制選字的用途有關聯，評估時仍要講清楚是在改報告的機率還是改生成方式。

softmax的「相對」可以從三候選看清：最高分2仍只比1、0高一些，所以第一項佔約三分之二；乘10後差距成10、20，第一項幾乎拿走所有比例。候選名稱與對錯完全沒有參與這個轉換。若三個候選全部不適合，softmax仍必須把1分配完，於是至少一項最高。開放問題還可能沒有把真正正確答案放進候選，這更說明最大比例不是可靠性的公式保證。

練習只把`correct_index`改成0，先預測信心數字完全不變，正確判定兩次都變True，再重跑核對。這個改變來自標準答案，不是模型能力變化。要判斷信心可靠性，必須有許多獨立題與真值，比較比如「信心0.9的題有多少真的正確」，下一節會把這件事算出來。
